In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch -q
# !pip install transformers -q
# !pip install matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess
import sys

def _install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_install("torch")
_install("transformers")
_install("matplotlib")
_install("numpy")

In [ ]:
import ast
import math
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Codex 代码实战：学习实现 vs 工程实现

基于论文 *Evaluating Large Language Models Trained on Code*（Chen et al., 2021），
用 **代码补全 / 多候选采样 / pass@k 评估** 任务演示 Codex 的核心思想。

本 Notebook 保留两条并行路径，并尽量使用 **同一类代码补全 Prompt** 做对照：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解温度、核采样、HumanEval、pass@k 与 decoder-only 生成逻辑 | 掌握工业级 `generate()`、批量推理、多候选采样与候选评估 |
| 实现方式 | 轻量 L2 / Hybrid：手写采样 + toy HumanEval + 微型解码器 | E1：`transformers` 预训练因果语言模型推理 |
| 代码量 | 较多，显式展示每一步计算与评估流程 | 较少，直接调用库 API |
| 适合场景 | 面试准备、原理拆解、教学演示 | 快速验证、工程原型、推理流程理解 |

> 两条路径不是两套无关的代码，而是同一套 Codex 思想在不同抽象层级上的表达。

## Section i：论文背景、任务定义与范围

### 1. 论文与任务
- **论文**：*Evaluating Large Language Models Trained on Code*（OpenAI, 2021）
- **核心模型**：Codex，本质上是经过代码数据进一步训练的 decoder-only 大语言模型
- **核心任务**：给定函数签名、注释或 docstring，生成能通过测试的代码

### 2. 为什么这个任务最能体现 Codex 的价值
Codex 真正改变行业的，不只是“模型会写代码”，而是它把代码生成任务从“像不像参考答案”改成了“能不能通过测试”。
因此，这份 Notebook 的主线不是复现大模型训练，而是拆开 Codex 最关键的四个环节：
1. **采样策略**：温度与核采样如何控制多样性
2. **评估标准**：为什么代码要看 functional correctness，而不是 BLEU
3. **pass@k**：为什么要一次生成多个候选
4. **解码器生成**：训练阶段与推理阶段到底有什么区别

### 3. 本 Notebook 覆盖什么，不覆盖什么
- **覆盖**：采样、pass@k、HumanEval 风格评估、微型 decoder-only 代码生成、工程化 `generate()`
- **不覆盖**：真实 Codex 的大规模训练数据、两阶段微调细节、完整 HumanEval 164 题复现
- **原因**：Codex 权重与大规模训练流程并不适合在免费 Colab / 本地 CPU 上稳定复现，但核心机制完全可以教学化拆开。

## Section ii：最小必要理论

### 1. 自回归代码生成

$$P(x_1, x_2, \dots, x_n) = \prod_{t=1}^{n} P(x_t \mid x_{<t})$$

含义：模型不是一次性输出整段代码，而是每一步根据已有前缀预测下一个 token。

### 2. 温度缩放

$$P(y_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- $T < 1$：分布更尖锐，更确定
- $T > 1$：分布更平坦，更多样

### 3. 核采样（Top-p Sampling）

先按概率从高到低排序，再取**累计概率刚超过 $p$ 的最小候选集合**，最后在这个集合里重新归一化并采样。

### 4. pass@k

$$\mathrm{pass@k} = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

其中：
- $n$：总候选数
- $c$：通过测试的候选数
- $k$：最终允许用户挑选的候选数

这说明：如果一次生成 100 个候选，只要其中有一个能通过测试，就算成功。

### 5. 为什么训练和推理不同
- **训练**：teacher forcing，模型看见完整正确前缀，所有位置并行算 loss
- **推理**：autoregressive generation，每次只取最后一个位置的 logits，再决定下一个 token

### 组件映射表（Mandatory）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Decoder-only Transformer | 微型解码器 `MiniCodex` | `AutoModelForCausalLM` | Runnable |
| Temperature Scaling | `temperature_softmax()` 手写实现 | `generate(temperature=...)` | Runnable |
| Nucleus Sampling | `nucleus_sampling()` 手写实现 | `generate(top_p=..., do_sample=True)` | Runnable |
| Functional Correctness | toy HumanEval 风格测试集 | 工程路径中的安全候选校验 | Runnable |
| pass@k | `pass_at_k()` 无偏估计 | 多候选生成后统一评估 | Runnable |
| Teacher Forcing | 微型解码器训练循环 | 真实大模型预训练阶段使用 | Illustrative |
| Autoregressive Inference | `generate_code_toy()` 手写循环 | `model.generate()` | Runnable |
| Batch Inference | 学习路径只做单条演示 | tokenizer 左侧 padding + `attention_mask` | Runnable |
| KV Cache | 只解释思想，不手写缓存逻辑 | `use_cache=True` / `past_key_values` | Explain-only |
| 两阶段代码专化训练 | 只在 markdown 解释 | 工程路径不复现训练，只做推理 | Explain-only |

## Section 1：数据准备

真实 Codex 使用的是大规模 GitHub 代码与后续指令 / 高质量数据训练，这里显然无法复现。
所以学习路径采用一个**可稳定运行的最小替代方案**：

1. 构造一个很小的 Python 函数语料，训练微型字符级 decoder-only 模型
2. 准备一组共享的代码补全 Prompt，供学习路径与工程路径共同使用
3. 构造一个 toy HumanEval 风格测试集，专门演示 functional correctness 与 pass@k

> 关键原则：我们不追求“训练出 Codex”，而追求“把 Codex 最重要的机制拆出来并跑通”。

In [ ]:
SHARED_PROMPTS = [
    "def add(a, b):\n    return ",
    "def is_even(n):\n    return ",
    "def reverse_string(s):\n    return ",
    "def max_two(a, b):\n    return ",
]

CODE_SAMPLES = [
    "def add(a, b):\n    return a + b",
    "def sub(a, b):\n    return a - b",
    "def mul(a, b):\n    return a * b",
    "def double(x):\n    return x * 2",
    "def triple(x):\n    return x * 3",
    "def negate(x):\n    return -x",
    "def square(x):\n    return x * x",
    "def cube(x):\n    return x * x * x",
    "def is_even(n):\n    return n % 2 == 0",
    "def is_odd(n):\n    return n % 2 != 0",
    "def is_positive(x):\n    return x > 0",
    "def is_negative(x):\n    return x < 0",
    "def max_two(a, b):\n    return a if a > b else b",
    "def min_two(a, b):\n    return a if a < b else b",
    "def reverse_string(s):\n    return s[::-1]",
    "def first_char(s):\n    return s[0] if s else ''",
    "def last_char(s):\n    return s[-1] if s else ''",
    "def concat(a, b):\n    return a + b",
    "def to_upper(s):\n    return s.upper()",
    "def to_lower(s):\n    return s.lower()",
]

TRAIN_TEXT = ("\n\n".join(CODE_SAMPLES) + "\n") * 40

class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.char2idx = {c: i + 2 for i, c in enumerate(chars)}
        self.char2idx["<PAD>"] = 0
        self.char2idx["<EOS>"] = 1
        self.idx2char = {v: k for k, v in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)

    def encode(self, text):
        return [self.char2idx.get(c, 0) for c in text] + [self.char2idx["<EOS>"]]

    def decode(self, ids):
        return "".join(self.idx2char[i] for i in ids if i not in (0, 1))

tokenizer_char = CharTokenizer(CODE_SAMPLES + SHARED_PROMPTS + [TRAIN_TEXT])
PAD_ID = tokenizer_char.char2idx["<PAD>"]
EOS_ID = tokenizer_char.char2idx["<EOS>"]
VOCAB_SIZE = tokenizer_char.vocab_size

print(f"Toy corpus size: {len(TRAIN_TEXT)} characters")
print(f"Vocabulary size: {VOCAB_SIZE}")
print("Shared prompts:")
for prompt in SHARED_PROMPTS:
    print("-", prompt.replace("\n", " "))

In [ ]:
MAX_LEN = 72
BATCH_SIZE = 32

class CodeDataset(Dataset):
    def __init__(self, text, tokenizer, seq_len=MAX_LEN):
        self.seq_len = seq_len
        self.data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.seq_len - 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]              # (seq_len,)
        y = self.data[idx + 1 : idx + self.seq_len + 1]      # (seq_len,)
        return x, y

dataset = CodeDataset(TRAIN_TEXT, tokenizer_char, MAX_LEN)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

x_batch, y_batch = next(iter(train_loader))
print(f"Input batch shape:  {tuple(x_batch.shape)}")
print(f"Target batch shape: {tuple(y_batch.shape)}")
print("\nDecoded sample input:\n")
print(tokenizer_char.decode(x_batch[0].tolist()))
print("\nDecoded sample target:\n")
print(tokenizer_char.decode(y_batch[0].tolist()))

## Section 2：共享组件与实现决策

### 可行性决策
- **学习路径深度**：L2 / Hybrid
  - 原因：Codex 的真正能力依赖大规模代码预训练，无法在免费 Colab 上完整复现。
  - 但温度、核采样、pass@k、toy HumanEval、微型 decoder-only 模型都可以单独跑通。
- **工程路径形式**：E1（成熟工业库）
  - 原因：真实工程使用方式就是直接加载预训练因果语言模型，再调用 `generate()`。
- **工程动作**：inference-only
  - 原因：本 Notebook 目标是理解 Codex 的使用与评估，不是复现预训练。

### 接下来会共用哪些组件
1. 共享超参数
2. 共享 Prompt 集合
3. 共享 pass@k 计算逻辑
4. 学习路径与工程路径的输出对照

In [ ]:
D_MODEL = 96
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 192
DROPOUT = 0.1
LR = 3e-3
TRAIN_STEPS = 60
TOP_P_DEFAULT = 0.9
MODEL_ID = "distilgpt2"

def temperature_softmax(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    scaled_logits = logits / temperature
    return F.softmax(scaled_logits, dim=-1)

def shannon_entropy(probs):
    probs = probs.clamp_min(1e-12)
    return float(-(probs * probs.log()).sum().item())

print({
    "D_MODEL": D_MODEL,
    "NUM_HEADS": NUM_HEADS,
    "NUM_LAYERS": NUM_LAYERS,
    "D_FF": D_FF,
    "LR": LR,
    "TRAIN_STEPS": TRAIN_STEPS,
    "MODEL_ID": MODEL_ID,
})

### 学习组件 A：Temperature Scaling

温度不是“额外加一个随机数”，而是先把 logits 进行缩放，再交给 softmax：

$$P(y_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

输入 / 输出 shape：
- logits：$(vocab\_size,)$
- probs：$(vocab\_size,)$

直觉上：
- 低温让概率质量更集中，适合追求确定性；
- 高温让分布更平，适合追求多样性；
- 这也是为什么 Codex 论文里 `pass@1` 与 `pass@100` 的最优温度并不一样。

In [ ]:
demo_logits = torch.tensor([4.0, 2.2, 1.4, 0.8, 0.1, -0.5], dtype=torch.float32)
demo_vocab = ["return a+b", "return a-b", "return a*b", "return 0", "pass", "raise"]

print("=== Temperature demo ===")
for temperature in [0.2, 0.5, 1.0, 1.5, 2.0]:
    probs = temperature_softmax(demo_logits, temperature)
    top_idx = int(probs.argmax().item())
    entropy = shannon_entropy(probs)
    print(
        f"T={temperature:<3} | top={demo_vocab[top_idx]:<12} | entropy={entropy:.3f} | "
        f"probs={[round(p, 3) for p in probs.tolist()]}"
    )

### 学习组件 B：Nucleus Sampling（Top-p Sampling）

核采样的目标不是“固定保留前 k 个 token”，而是：
1. 按概率排序；
2. 找到累计概率刚超过阈值 $p$ 的最小集合；
3. 只在这个集合中重新归一化并采样。

这样做的直觉是：
- 当模型很确定时，候选集合会自动变小；
- 当模型不确定时，候选集合会自动变大。

这比固定 `top_k` 更符合代码生成任务中“按置信度调节搜索空间”的需求。

In [ ]:
def nucleus_sampling(logits, temperature=1.0, top_p=0.9):
    probs = temperature_softmax(logits, temperature)                     # (vocab_size,)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)    # (vocab_size,)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)                # (vocab_size,)

    remove_mask = cumulative_probs > top_p
    remove_mask = torch.roll(remove_mask, shifts=1, dims=0)
    remove_mask[0] = False

    filtered_probs = sorted_probs.masked_fill(remove_mask, 0.0)
    filtered_probs = filtered_probs / filtered_probs.sum()
    sampled_idx = torch.multinomial(filtered_probs, num_samples=1)
    token_id = sorted_indices[sampled_idx]
    return int(token_id.item())

def nucleus_size(logits, temperature=1.0, top_p=0.9):
    probs = temperature_softmax(logits, temperature)
    sorted_probs, _ = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    remove_mask = cumulative_probs > top_p
    remove_mask = torch.roll(remove_mask, shifts=1, dims=0)
    remove_mask[0] = False
    return int((~remove_mask).sum().item())

print("=== Nucleus sampling demo ===")
for top_p in [0.3, 0.5, 0.8, 0.95, 1.0]:
    samples = [demo_vocab[nucleus_sampling(demo_logits, temperature=0.9, top_p=top_p)] for _ in range(25)]
    print(f"top_p={top_p:<4} | nucleus_size={nucleus_size(demo_logits, 0.9, top_p):<2} | counts={dict(Counter(samples))}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x = np.arange(len(demo_logits))
width = 0.18
for i, temperature in enumerate([0.2, 0.5, 1.0, 1.5]):
    probs = temperature_softmax(demo_logits, temperature).numpy()
    axes[0].bar(x + i * width, probs, width=width, alpha=0.85, label=f"T={temperature}")

axes[0].set_title("Temperature vs Probability")
axes[0].set_xlabel("Candidate token")
axes[0].set_ylabel("Probability")
axes[0].set_xticks(x + 1.5 * width)
axes[0].set_xticklabels([f"c{i}" for i in range(len(demo_logits))])
axes[0].legend()
axes[0].grid(alpha=0.2)

p_values = np.linspace(0.1, 1.0, 19)
for temperature in [0.5, 1.0, 1.5]:
    sizes = [nucleus_size(demo_logits, temperature, p) for p in p_values]
    axes[1].plot(p_values, sizes, marker="o", markersize=3, label=f"T={temperature}")

axes[1].set_title("Top-p vs Nucleus Size")
axes[1].set_xlabel("top_p")
axes[1].set_ylabel("Nucleus size")
axes[1].grid(alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()

## Section iii：Functional Correctness 与 pass@k

### 为什么代码评估不能只看文本相似度？
对自然语言任务，BLEU 或 ROUGE 还能粗略反映“像不像标准答案”；
但代码任务里：
- 两段功能等价代码，文本可以差异很大；
- 两段看起来几乎一样的代码，也可能因为一个边界错误就完全失败。

因此 Codex 论文强调 **functional correctness**：
> 不问它像不像参考解，只问它能不能通过测试。

下面会先实现 `pass@k`，再用一个 toy HumanEval 风格题集把“多候选生成 → 测试 → 汇总指标”跑通。

In [ ]:
def pass_at_k(n, c, k):
    if n <= 0:
        raise ValueError("n must be positive")
    if not (0 <= c <= n):
        raise ValueError("c must satisfy 0 <= c <= n")
    if not (1 <= k <= n):
        raise ValueError("k must satisfy 1 <= k <= n")
    if n - c < k:
        return 1.0

    log_ratio = 0.0
    for i in range(k):
        log_ratio += math.log(n - c - i) - math.log(n - i)
    return 1.0 - math.exp(log_ratio)

print("=== pass@k demo ===")
for n, c, k in [(100, 10, 1), (100, 10, 10), (100, 10, 50), (200, 5, 1), (200, 5, 20)]:
    print(f"n={n:<3} c={c:<3} k={k:<3} -> pass@k={pass_at_k(n, c, k):.4f}")

### HumanEval 风格的最小评测集

真实 HumanEval 有 164 道 Python 函数题。这里不追求规模，而是保留它最关键的评测思想：
- 给定函数签名与说明
- 生成函数体
- 用单元测试验证功能正确性

> 安全说明：下面的 `execute_trusted_candidate()` 只执行 **Notebook 内部写死的 toy 候选代码**，用于教学演示。后面的工程路径不会自动执行任意开放式模型生成代码。

In [ ]:
PROBLEMS = [
    {
        "task_id": "double",
        "prompt": "def double(x):\n",
        "test_cases": [
            "assert double(3) == 6",
            "assert double(0) == 0",
            "assert double(-2) == -4",
        ],
        "canonical_solution": "    return x * 2\n",
        "wrong_solutions": [
            "    return x + 2\n",
            "    return x\n",
            "    return 2\n",
            "    return -x\n",
        ],
    },
    {
        "task_id": "negate",
        "prompt": "def negate(x):\n",
        "test_cases": [
            "assert negate(5) == -5",
            "assert negate(-3) == 3",
            "assert negate(0) == 0",
        ],
        "canonical_solution": "    return -x\n",
        "wrong_solutions": [
            "    return x\n",
            "    return abs(x)\n",
            "    return x * 2\n",
            "    return 0\n",
        ],
    },
    {
        "task_id": "add",
        "prompt": "def add(a, b):\n",
        "test_cases": [
            "assert add(2, 3) == 5",
            "assert add(-1, 4) == 3",
            "assert add(0, 0) == 0",
        ],
        "canonical_solution": "    return a + b\n",
        "wrong_solutions": [
            "    return a - b\n",
            "    return a\n",
            "    return b\n",
            "    return a * b\n",
        ],
    },
]

def execute_trusted_candidate(code_str, test_cases):
    """仅执行 Notebook 内部写死的 toy 候选代码。"""
    try:
        exec_globals = {}
        exec(code_str, exec_globals)
        for test in test_cases:
            exec(test, exec_globals)
        return True, None
    except Exception as e:
        return False, str(e)

print("=== Verify canonical toy solutions ===")
for problem in PROBLEMS:
    code = problem["prompt"] + problem["canonical_solution"]
    passed, error = execute_trusted_candidate(code, problem["test_cases"])
    print(problem["task_id"], "PASS" if passed else f"FAIL: {error}")

### 完整的 toy pass@k 流水线

真实 Codex 会：
1. 对同一道题生成多个候选；
2. 执行测试；
3. 统计通过数量；
4. 用无偏公式得到 pass@k。

这里我们用一个**可控的候选池**来模拟不同模型能力：
- `correct_rate` 越高，表示模型越容易采样到正确候选；
- `k` 越大，表示用户可以从更多候选里挑一个。

In [ ]:
rng = np.random.default_rng(42)

def sample_candidates(problem, n_samples=50, correct_rate=0.1):
    results = []
    candidates = []
    for _ in range(n_samples):
        if rng.random() < correct_rate:
            body = problem["canonical_solution"]
        else:
            body = rng.choice(problem["wrong_solutions"])
        code = problem["prompt"] + body
        passed, _ = execute_trusted_candidate(code, problem["test_cases"])
        candidates.append(code)
        results.append(passed)
    return candidates, results

def evaluate_problem_set(problems, n_samples=80, correct_rate=0.1, k_values=(1, 5, 10, 20)):
    summary = {}
    for k in k_values:
        scores = []
        for problem in problems:
            _, results = sample_candidates(problem, n_samples=n_samples, correct_rate=correct_rate)
            scores.append(pass_at_k(len(results), sum(results), k))
        summary[k] = float(np.mean(scores))
    return summary

correct_rates = [0.05, 0.10, 0.20, 0.40]
passk_results_by_rate = {}
print("=== Toy pass@k evaluation ===")
print(f"{'correct_rate':>12} {'pass@1':>10} {'pass@5':>10} {'pass@10':>10} {'pass@20':>10}")
print("-" * 58)
for rate in correct_rates:
    summary = evaluate_problem_set(PROBLEMS, n_samples=80, correct_rate=rate)
    passk_results_by_rate[rate] = summary
    print(f"{rate:>12.0%} {summary[1]:>10.4f} {summary[5]:>10.4f} {summary[10]:>10.4f} {summary[20]:>10.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

k_range = list(range(1, 41))
for rate in [0.05, 0.10, 0.20, 0.40]:
    c = int(80 * rate)
    c = max(c, 1)
    curve = [pass_at_k(80, c, k) for k in k_range]
    axes[0].plot(k_range, curve, linewidth=2, label=f"correct_rate={rate:.0%}")

axes[0].set_title("pass@k vs Candidate Budget")
axes[0].set_xlabel("k")
axes[0].set_ylabel("pass@k")
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.2)
axes[0].legend()

for k in [1, 5, 10, 20]:
    y = [passk_results_by_rate[rate][k] for rate in correct_rates]
    axes[1].plot(correct_rates, y, marker="o", linewidth=2, label=f"pass@{k}")

axes[1].set_title("Model Quality vs pass@k")
axes[1].set_xlabel("Underlying correct rate")
axes[1].set_ylabel("pass@k")
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()

## Section iv：学习路径（微型 decoder-only 代码生成器）

到这里，我们已经把 Codex 的“采样与评估”拆开了。
接下来补上最底层的**生成器视角**：

- 使用一个极小的 decoder-only Transformer
- 通过 teacher forcing 做短时间训练
- 在推理阶段手写自回归循环

这里不追求模型强度，而是追求两个教学目标：
1. 看懂训练时 logits 和 loss 是怎么来的；
2. 看懂工程路径 `generate()` 背后到底封装了什么。

In [ ]:
class MiniCodex(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.token_emb = nn.Embedding(vocab_size, d_model)                     # (vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)                          # (max_len, d_model)
        self.drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight                             # weight tying

    def causal_mask(self, seq_len, device):
        return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, idx):
        B, T = idx.shape                                                        # (batch, seq)
        pos = torch.arange(T, device=idx.device).unsqueeze(0)                   # (1, seq)
        x = self.token_emb(idx) + self.pos_emb(pos)                             # (batch, seq, d_model)
        x = self.drop(x)
        x = self.transformer(x, mask=self.causal_mask(T, idx.device))           # (batch, seq, d_model)
        x = self.ln_f(x)                                                         # (batch, seq, d_model)
        logits = self.lm_head(x)                                                 # (batch, seq, vocab_size)
        return logits

toy_model = MiniCodex(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, MAX_LEN, DROPOUT).to(device)
with torch.no_grad():
    toy_model_logits = toy_model(x_batch[:2].to(device))
print(f"Toy model logits shape: {tuple(toy_model_logits.shape)}")
print(f"Toy model params: {sum(p.numel() for p in toy_model.parameters()):,}")

### 训练 vs 推理的区别（Learning Path）

#### 训练阶段
- 输入完整前缀序列 `x`
- 目标是右移一位后的 `y`
- 所有位置同时计算交叉熵损失
- 这就是 **teacher forcing**

#### 推理阶段
- 只取最后一个位置的 logits
- 用 greedy / temperature / nucleus 选下一个 token
- 把新 token 拼回输入再继续
- 这就是 **autoregressive generation**

下面先训练一个很小的 toy 模型，再展示三种推理策略。

In [ ]:
optimizer = torch.optim.AdamW(toy_model.parameters(), lr=LR)
loss_history = []

toy_model.train()
train_iter = iter(train_loader)
for step in range(1, TRAIN_STEPS + 1):
    try:
        xb, yb = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        xb, yb = next(train_iter)

    xb = xb.to(device)
    yb = yb.to(device)
    logits = toy_model(xb)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(toy_model.parameters(), 1.0)
    optimizer.step()

    loss_history.append(float(loss.item()))
    if step == 1 or step % 10 == 0:
        print(f"Step {step:02d}/{TRAIN_STEPS} | Loss: {loss.item():.4f}")

plt.figure(figsize=(6.5, 4.2))
plt.plot(range(1, TRAIN_STEPS + 1), loss_history)
plt.title("Toy Decoder Training Loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def generate_code_toy(model, tokenizer, prompt, max_new_tokens=32, strategy="nucleus", temperature=0.8, top_p=0.9):
    model.eval()
    ids = tokenizer.encode(prompt)[:-1]
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = ids[:, -MAX_LEN:]
        logits = model(idx_cond)
        next_logits = logits[0, -1, :]

        if strategy == "greedy":
            next_id = int(next_logits.argmax().item())
        elif strategy == "temperature":
            probs = temperature_softmax(next_logits, temperature)
            next_id = int(torch.multinomial(probs, 1).item())
        elif strategy == "nucleus":
            next_id = nucleus_sampling(next_logits, temperature=temperature, top_p=top_p)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
        if next_id == EOS_ID:
            break

    full_text = tokenizer.decode(ids[0].tolist())
    continuation = full_text[len(prompt):]
    return continuation

toy_outputs = {}
print("=== Toy decoder generation ===")
for prompt in SHARED_PROMPTS:
    toy_outputs[prompt] = {}
    print(f"\nPrompt: {prompt.strip()}")
    for strategy in ["greedy", "temperature", "nucleus"]:
        continuation = generate_code_toy(toy_model, tokenizer_char, prompt, strategy=strategy, temperature=0.8, top_p=TOP_P_DEFAULT)
        toy_outputs[prompt][strategy] = continuation
        print(f"[{strategy:>11}] {continuation[:80]!r}")

## Section vi：工程路径 + 对比 + 面试表达

### 工程路径（E1：成熟库 + inference-only）
真实 Codex 已不可直接在本 Notebook 中加载，因此工程路径采用一个**更轻量、但接口形态一致**的因果语言模型：`distilgpt2`。

工程关注点：
- `AutoTokenizer`
- `AutoModelForCausalLM`
- `model.generate()`
- 左侧 padding、`attention_mask`、`pad_token_id`
- 多候选生成与后处理

### 黑盒拆解：`generate()` 到底做了什么？
1. tokenizer 把 prompt 变成 `input_ids` 和 `attention_mask`
2. 模型前向传播得到最后一个位置的 logits
3. 按 greedy / sampling / beam search 选择下一个 token
4. 把新 token 拼回输入继续循环
5. 必要时使用 `use_cache` / `past_key_values` 加速

### 学习路径 vs 工程路径对比（Mandatory）

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 明确看到温度、核采样、pass@k、teacher forcing、自回归循环 | 明确看到工业级 tokenizer / model / generate / batch 推理接口 |
| 代码量 | 较多，几乎每一步都显式展开 | 较少，调用高层 API 即可完成 |
| 灵活性 | 高：可以改采样、改模型、改评估逻辑 | 中：主要受库接口限制 |
| 稳定性 | 中：toy 训练容易受数据规模影响 | 高：预训练模型与成熟 API 更稳 |
| 工业适配度 | 适合教学、面试、机制实验 | 适合原型验证、批量生成、候选筛选 |
| 适用场景 | 理解 Codex 的关键思想、解释 pass@k、拆解生成机制 | 快速接入代码补全、构建多候选流水线、工程评估 |

### 训练 vs 推理差异（两条路径都要说清楚）

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | toy `MiniCodex` 用 teacher forcing，输入完整前缀并计算全部位置 loss | 本 Notebook 不训练 `distilgpt2`；真实大模型训练在离线阶段完成 |
| 推理 | `generate_code_toy()` 每次取最后一个 logits，再选一个 token 拼回输入 | `model.generate()` 封装了同样的循环 |
| 采样 | 显式调用 `temperature_softmax()` / `nucleus_sampling()` | 通过 `temperature`、`top_p`、`do_sample` 参数控制 |
| mask | 学习路径里显式构造 causal mask | 工程路径里由模型内部自动处理 |
| padding | toy 单条生成几乎不强调 padding | 批量推理时左侧 padding + `attention_mask` 很重要 |
| cache | 只解释思想，不手写缓存 | `use_cache=True` 与 `past_key_values` 用于加速 |

### 结果边界与适用条件
- **学习路径**证明的是：即使不复现真实 Codex 训练，我们仍然能把它最关键的采样、评估、生成逻辑拆出来并跑通。
- **工程路径**展示的是：真实生产环境里，大家不会手写整个解码器，而是直接使用成熟库完成批量推理、多候选生成与候选过滤。
- **范围边界**：这里不复现真实 Codex 权重、训练语料与完整 HumanEval。
- **已知限制**：toy 模型只用于机制演示；工程路径使用 `distilgpt2`，只能近似展示接口与流程，不能代表真实 Codex 能力上限。
- **安全边界**：Notebook 不自动执行任意开放式模型生成代码；工程路径只做批量生成与安全的语法层过滤演示。

## 面试与项目表达（Mandatory）

### 高频面试题

**Q1：为什么 Codex 要强调 pass@k，而不是只看 pass@1？**
- 因为代码生成有随机性，一次生成失败不代表模型没有能力。
- 在真实开发里，工程师也常常愿意从多个候选里挑一个可用版本。
- pass@k 更符合“多候选辅助编程”的真实使用方式。

**Q2：为什么代码评估不能用 BLEU 这类文本指标代替功能测试？**
- 功能等价代码可以写得完全不同。
- 代码只要有一个边界错误，文本可能非常像，但功能完全错误。
- 所以代码评估要优先看 functional correctness。

**Q3：温度和 top_p 分别控制什么？**
- 温度控制分布尖锐程度。
- top_p 控制候选集覆盖的累计概率质量。
- 二者结合起来控制“多样性”和“稳定性”的平衡。

**Q4：为什么 `pass@1` 和 `pass@100` 的最优温度可能不同？**
- `pass@1` 更看重单次输出的确定性，因此偏低温。
- `pass@100` 更看重候选多样性，因此适度更高温更有利。

**Q5：`model.generate()` 本质上做了什么？**
- 前向传播得到最后一个位置 logits；
- 按解码策略选 token；
- 拼回输入；
- 重复直到结束条件满足。

**Q6：为什么 decoder-only 模型批量生成时常用左侧 padding？**
- 因为模型是从右端继续生成的。
- 左侧 padding 更符合其训练 / 推理习惯，避免把 padding 误当成可续写内容。

**Q7：`use_cache` / `past_key_values` 的作用是什么？**
- 把历史 key/value 缓存下来，避免每次生成新 token 都重复计算旧上下文。
- 序列越长，加速效果越明显。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | Codex 的核心贡献是什么？ | 不只是会写代码，而是把代码评估标准推进到 functional correctness + pass@k。 |
| 2 | 为什么要多候选生成？ | 因为代码生成有随机性，多候选能显著提高至少一个可用解出现的概率。 |
| 3 | 温度和 top_p 怎么配合？ | 温度调分布形状，top_p 调候选覆盖范围。 |
| 4 | `generate()` 本质是什么？ | 封装好的自回归推理循环。 |
| 5 | 为什么代码任务看测试不看 BLEU？ | 因为代码正确性首先是功能问题，不是表面文本相似度问题。 |
| 6 | 为什么要左侧 padding？ | 因为 decoder-only 模型从最右侧位置继续生成。 |

### 项目表达

> 如果面试官问“你做过什么相关项目”，可以这样组织回答：

- **学习深度**：我拆过 Codex 相关的温度缩放、核采样、pass@k、HumanEval 风格评估，以及一个微型 decoder-only 代码生成器。
- **工程能力**：我用 `transformers` 搭过代码补全与多候选生成流程，处理过 `generate()`、批量推理、padding、候选过滤。
- **对比思考**：我能说清楚为什么代码任务需要 functional correctness，为什么 `pass@1` 和 `pass@k` 关注点不同，也能解释学习版和工业版之间的对应关系。

### 延伸阅读与对比

| 对比维度 | Codex | AlphaCode | CodeLlama |
|---|---|---|---|
| 代表问题 | 代码补全与通用编程 | 竞赛型程序合成 | 开源代码大模型 |
| 候选策略 | 多候选采样 + pass@k | 超大量候选 + 过滤 | 多种解码策略 |
| 核心评估 | HumanEval / pass@k | 竞赛题通过率 | HumanEval / 工程基准 |
| 工程启发 | 候选生成 + 测试筛选 | 大规模搜索与重排序 | 开源部署与定制 |

### 进阶探索方向
- 对比 `top_k`、`top_p`、beam search 在代码任务上的效果差异
- 引入更严格的安全沙箱来执行候选代码
- 用更强的开源代码模型替换 `distilgpt2`，观察 pass@k 与候选质量变化
- 系统研究“生成 → 静态检查 → 单测 → 重排序”的完整工程流水线

## Appendix A：补充说明
- 这里选择 **L2 / Hybrid**，因为 Codex 真正有价值的部分不在于从零训练一个小模型，而在于多候选采样、functional correctness 与 pass@k。
- 这份 Notebook 最适合：准备代码大模型面试、想理解 `generate()` 与 `pass@k`、想从“会调 API”进一步走到“知道为什么这样设计”的读者。

In [ ]:
tokenizer_hf = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer_hf.pad_token is None:
    tokenizer_hf.pad_token = tokenizer_hf.eos_token

model_hf = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
model_hf.eval()

print(f"Loaded model: {MODEL_ID}")
print(f"Tokenizer vocab size: {tokenizer_hf.vocab_size:,}")
print(f"Model params: {sum(p.numel() for p in model_hf.parameters()):,}")
print(f"use_cache default: {model_hf.config.use_cache}")

def generate_hf(prompts, max_new_tokens=24, do_sample=True, temperature=0.8, top_p=0.95, num_return_sequences=1, num_beams=1):
    inputs = tokenizer_hf(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(device)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        num_beams=num_beams,
        pad_token_id=tokenizer_hf.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p)

    with torch.no_grad():
        outputs = model_hf.generate(**generation_kwargs)

    return tokenizer_hf.batch_decode(outputs, skip_special_tokens=True)

print("\n=== Engineering generation demo ===")
for prompt in SHARED_PROMPTS[:2]:
    greedy = generate_hf([prompt], do_sample=False, num_beams=1)[0]
    sampled = generate_hf([prompt], do_sample=True, temperature=0.8, top_p=0.95)[0]
    beam = generate_hf([prompt], do_sample=False, num_beams=3)[0]
    print(f"\nPrompt: {prompt.strip()}")
    print(f"[greedy]   {greedy[:120]}")
    print(f"[sampling] {sampled[:120]}")
    print(f"[beam]     {beam[:120]}")

batched_outputs = generate_hf(SHARED_PROMPTS, max_new_tokens=20, do_sample=True, temperature=0.8, top_p=0.95)
print("\n=== Batched engineering outputs ===")
for prompt, output in zip(SHARED_PROMPTS, batched_outputs):
    print(f"\nPrompt: {prompt.strip()}")
    print(output[:160])

def is_valid_python_completion(full_code: str):
    try:
        ast.parse(full_code)
        return True
    except SyntaxError:
        return False

SAFE_ENGINEERING_PROMPTS = [
    "def double(x):\n    return ",
    "def add(a, b):\n    return ",
]

print("\n=== Safe engineering candidate filtering ===")
for prompt in SAFE_ENGINEERING_PROMPTS:
    candidates = generate_hf([prompt], max_new_tokens=12, do_sample=True, temperature=0.8, top_p=0.95, num_return_sequences=8)
    syntax_ok = 0
    shown = 0
    for candidate in candidates:
        completion = candidate[len(prompt):]
        full_code = prompt + completion
        ok = is_valid_python_completion(full_code)
        syntax_ok += int(ok)
        if shown < 3:
            status = "PARSE_OK" if ok else "SYNTAX_ERR"
            print(f"Prompt: {prompt.strip()} | {status:<10} | {completion[:80]!r}")
            shown += 1

    parse_pk1 = pass_at_k(len(candidates), syntax_ok, 1)
    parse_pk3 = pass_at_k(len(candidates), syntax_ok, 3)
    parse_pk5 = pass_at_k(len(candidates), syntax_ok, 5)
    print(f"Summary -> parseable={syntax_ok}/{len(candidates)}, parse@1={parse_pk1:.3f}, parse@3={parse_pk3:.3f}, parse@5={parse_pk5:.3f}\n")

learning_curve = [passk_results_by_rate[rate][5] for rate in correct_rates]
engineering_demo_scores = [min(1.0, rate * 2.2) for rate in correct_rates]  # illustrative only

plt.figure(figsize=(6.5, 4.2))
plt.plot(correct_rates, learning_curve, marker="o", linewidth=2, label="Learning path toy pass@5")
plt.plot(correct_rates, engineering_demo_scores, marker="s", linewidth=2, label="Engineering path illustrative curve")
plt.title("Learning vs Engineering Illustration")
plt.xlabel("Underlying generation quality")
plt.ylabel("Illustrative score")
plt.ylim(0, 1.05)
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

print("=== Side-by-side prompt outputs ===")
for prompt in SHARED_PROMPTS:
    toy_nucleus = toy_outputs[prompt]["nucleus"]
    hf_sample = generate_hf([prompt], max_new_tokens=20, do_sample=True, temperature=0.8, top_p=0.95)[0]
    print(f"\nPrompt: {prompt.strip()}")
    print(f"Learning path:   {toy_nucleus[:140]!r}")
    print(f"Engineering path:{hf_sample[:140]!r}")